In [11]:
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
data = pd.read_csv(r'..\Data\who_suicide_statistics.csv')
print("Data Information")
print(data.info())
print("#" * 50 + "\n" + "Data Description")
print(data.describe())
print("#" * 50 + "\n" + "First Five Rows")
print(data.head())

Data Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43776 entries, 0 to 43775
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   country      43776 non-null  object 
 1   year         43776 non-null  int64  
 2   sex          43776 non-null  object 
 3   age          43776 non-null  object 
 4   suicides_no  41520 non-null  float64
 5   population   38316 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 2.0+ MB
None
##################################################
Data Description
               year   suicides_no    population
count  43776.000000  41520.000000  3.831600e+04
mean    1998.502467    193.315390  1.664091e+06
std       10.338711    800.589926  3.647231e+06
min     1979.000000      0.000000  2.590000e+02
25%     1990.000000      1.000000  8.511275e+04
50%     1999.000000     14.000000  3.806550e+05
75%     2007.000000     91.000000  1.305698e+06
max     2016.000000 

In [13]:
# 去缺失值
print(data.isnull().sum())
data = data.dropna()
print(data.head(40))

# 构造自杀率列(百分比)
data['suicide_rate'] = data['suicides_no'] / data['population'] * 100

country           0
year              0
sex               0
age               0
suicides_no    2256
population     5460
dtype: int64
    country  year     sex          age  suicides_no  population
24  Albania  1987  female  15-24 years         14.0    289700.0
25  Albania  1987  female  25-34 years          4.0    257200.0
26  Albania  1987  female  35-54 years          6.0    278800.0
27  Albania  1987  female   5-14 years          0.0    311000.0
28  Albania  1987  female  55-74 years          0.0    144600.0
29  Albania  1987  female    75+ years          1.0     35600.0
30  Albania  1987    male  15-24 years         21.0    312900.0
31  Albania  1987    male  25-34 years          9.0    274300.0
32  Albania  1987    male  35-54 years         16.0    308000.0
33  Albania  1987    male   5-14 years          0.0    338200.0
34  Albania  1987    male  55-74 years          1.0    137500.0
35  Albania  1987    male    75+ years          1.0     21800.0
36  Albania  1988  female  15-24 ye

### 思路
* 自杀率水平与性别是否有统计上的显著性差异
* 不同年龄组的自杀率是否有显著性差异
* 哪些年龄组的自杀率有显著性差异
* 国家视角
* 自杀率时间做线性回归

### 自杀率与性别的统计推断

In [14]:
# Welch v
def get_df(SX: float, n: float, SY: float, m: float) -> int:
    '''
    返回估计的自由度
    SX为X组的样本标准差，n为该组的样本大小
    SY为Y组的样本标准差，m为该组的样本大小
    '''
    df = ((SX**2/n) + (SY**2/m))**2 / ((SX**2/n)**2/(n-1) + (SY**2/m)**2/(m-1))
    return int(df) if df - int(df) < 0.5 else int(df) + 1

In [ ]:
### Two-sample t-test

# significance level
a = 0.05

# feature cloumn
female_suicides = data.loc[data['sex'] == 'female', 'suicide_rate']
male_suicides = data.loc[data['sex'] == 'male', 'suicide_rate']

# sample size
female_size = female_suicides.size
male_size = male_suicides.size
print('\n', 'sample size', '\n', '#' * 50)
print(f'Female sample size: {female_size}')
print(f'Male sample size: {male_size}')

# sample mean
female_mean = female_suicides.mean()
male_mean = male_suicides.mean()
print('\n', 'sample mean', '\n', '#' * 50)
print(f'Female sample mean: {female_mean}')
print(f'Male sample mean: {male_mean}')

# sample variance
female_std = female_suicides.var() ** 0.5
male_std = male_suicides.var() ** 0.5
print('\n', 'sample variance', '\n', '#' * 50)
print(f'Female sample standard deviation: {female_std}')
print(f'Male sample standard deviation: {male_std}')

# df
df = get_df(female_std, female_size, male_std, male_size)

# test statistics
mean_diff = female_mean - male_mean
standard_error = (female_std**2 / female_size + male_std**2 / male_size) ** 0.5
t = mean_diff / standard_error

# p-value
p_value = 2 * stats.t.sf(abs(t), df=df)

# result
print('\n', 'Conclusion', '\n', '#' * 50)
if p_value <= a:
    print(f'At significance level a={a}, we can reject the null hypothesis')
else:
    print(f'At significance level a={a}, we fail to reject the null hypothesis')


 sample size 
 ##################################################
Female sample size: 18030
Male sample size: 18030

 sample mean 
 ##################################################
Female sample mean: 0.0057405545984963065
Male sample mean: 0.020629632846979457

 sample variance 
 ##################################################
Female sample standard deviation: 0.008215420181390258
Male sample standard deviation: 0.02477709824032179

 Conclusion 
 ##################################################
At significance level a=0.05, we can reject the null hypothesis


In [ ]:
# scipy官方实现，equal_var=False 表示Welch检验
t_scipy, p_scipy = stats.ttest_ind(female_suicides, male_suicides, equal_var=False)

print('\n', 'scipy官方结果验证', '\n', '#' * 50)
print(f'scipy t值: {t_scipy}')
print(f'scipy p值: {p_scipy}')


 scipy官方结果验证 
 ##################################################
scipy t值: -76.58881247585184
scipy p值: 0.0
